# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print out the title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Note: All entities must be referenced by their `@id`. We'll inspect available record sets, fields, and columns, and list their `@id`s and names for further use.

In [ ]:
# List all record sets in the dataset metadata
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in the Croissant metadata.")
else:
    print("Available Record Sets and Fields:")
    for rs in record_sets:
        print(f"- Record Set @id: {rs.id}, name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"      - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        print("  Columns:")
        if hasattr(rs, 'columns'):
            for column in rs.columns:
                print(f"      - Column @id: {column.id}, name: {column.name}")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll collect all record sets and load their data into a dictionary of DataFrames, accessible by their `@id`.

In [ ]:
# Prepare list of record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load data for each record set as DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with {len(df)} records. Columns: {df.columns.tolist()}")

# For demonstration, pick the first record set if available
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll choose a numeric field from the first record set for demonstration.

In [ ]:
# Select a numeric field for EDA
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    numeric_field_id = None
    # Try to auto-infer a numeric field
    record_sets = dataset.metadata.record_sets
    fields_map = {}
    for rs in record_sets:
        if rs.id == main_record_set_id:
            for field in rs.fields:
                fields_map[field.id] = field
                if field.data_type in ['Float', 'Integer', 'Number']:
                    numeric_field_id = field.id
                    break
    if numeric_field_id is None:
        # Fallback: Try to auto-detect in DataFrame
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    print(f"Numeric field selected for EDA: {numeric_field_id}")

    if numeric_field_id is not None:
        # Filter: greater than mean (if enough data)
        colname = numeric_field_id
        if colname in df.columns:
            threshold = df[colname].mean() if df[colname].notnull().sum() > 0 else 0
            filtered_df = df[df[colname] > threshold]
            print(f"Filtered records with {colname} > {threshold:.2f}:")
            print(filtered_df.head())

            # Normalize field
            filtered_df = filtered_df.copy()
            filtered_df[f"{colname}_normalized"] = (filtered_df[colname] - filtered_df[colname].mean()) / filtered_df[colname].std()
            print(f"Normalized {colname} for filtered records:")
            print(filtered_df[[colname, f"{colname}_normalized"].copy()].head())

            # Try to group by a categorical field
            group_field = None
            for field in fields_map.values():
                if field.data_type in ['Text']:
                    if field.id in filtered_df.columns:
                        group_field = field.id
                        break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[colname].mean().reset_index()
                print(f"Grouped data by {group_field} (mean):")
                print(grouped_df.head())
        else:
            print(f"Numeric field {colname} not found in DataFrame columns.")
    else:
        print("No numeric field detected for this record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field selected above (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Croissant schema allowed programmatic access to data structure and records using the `mlcroissant` library.
- Record sets and fields can be referenced by their `@id`s for consistent data processing.
- This notebook demonstrated standardized loading, cleaning, exploration, and visualization for the FAIR^2 dataset.